# Crisis Mining V10 (Colab)

Policy probes find failures, deep search solves them.
Model: 2W2 epoch 10. Uses **feature-value MCTS** (NN value head is broken
on val_weight=0 models — see HISTORY lesson 112-114).

**Critical:** BATCH_SIZE=8 — anything larger destroys MCTS quality
via virtual-loss starvation (HISTORY lesson 115).

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `pillar2w2_epoch_10.pt` — model
3. `feature_value_weights.npz` — feature-value evaluator

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
!cp {DRIVE}/pillar2w2_epoch_10.pt /content/alphatrain/data/model.pt
!cp {DRIVE}/feature_value_weights.npz /content/alphatrain/data/feature_value_weights.npz
print(f'Model: {os.path.getsize("/content/alphatrain/data/model.pt")/1e6:.0f} MB')
print(f'Feature weights: {os.path.getsize("/content/alphatrain/data/feature_value_weights.npz")} bytes')

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# === CRISIS MINING V10 BATCH 1 ===
# Instance 1: 700000-710000  (10K probes)
# Instance 2: 710000-720000  (10K probes)
SEED_START = 700000
SEED_END = 710000
WORKERS = 20
BATCH_SIZE = 8           # MCTS quality requires bs=8 (HISTORY lesson 115)
SAVE_DIR = f'{DRIVE}/crisis_v10'
# =============================

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Seeds: {SEED_START}-{SEED_END-1} ({SEED_END-SEED_START} probes)')

!python -m alphatrain.scripts.crisis_mining \
    --model /content/alphatrain/data/model.pt \
    --feature-value-weights /content/alphatrain/data/feature_value_weights.npz \
    --seed-start {SEED_START} --seed-end {SEED_END} \
    --recovery-turns 25 --recovery-sims 2000 \
    --prevention-turns 75 --prevention-sims 1600 \
    --continue-turns 500 \
    --workers {WORKERS} --device cuda --batch-size {BATCH_SIZE} \
    --compile \
    --save-dir {SAVE_DIR}